# Benchmark Gray iterE inference (CPU vs GPU)

**Kernel:** `conda env:.conda-diffusion` on EAF (for GPU). CPU also works on sbndbuild.

Times DDIM→DDIM reconstruction with Gray's EMA checkpoint
`emabrats2update_0.9999_111000.pt` (same path as production ROC/inference).

Writes JSON under `/exp/sbnd/data/users/munjung/anomaly-detection/training/benchmarks/`.


In [ ]:
from __future__ import annotations
import subprocess, sys
from pathlib import Path

APP = Path("/exp/sbnd/app/users/munjung/anomaly-detection")
SCRIPT = APP / "inference" / "benchmark_gputnam_inference.py"
OUT = Path("/exp/sbnd/data/users/munjung/anomaly-detection/training/benchmarks")
OUT.mkdir(parents=True, exist_ok=True)

import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
    print("CUDA_VISIBLE_DEVICES=", __import__("os").environ.get("CUDA_VISIBLE_DEVICES"))


In [ ]:
# Production-like settings: T=200 (and T=100), 1 patch, batch 1
cmd = [
    sys.executable,
    str(SCRIPT),
    "--device", "auto",   # cpu + cuda when available
    "--T", "100",
    "--T", "200",
    "--warmup", "1",
    "--repeats", "3",
    "--batch-size", "1",
    "--n-patches", "1",
    "--out-dir", str(OUT),
]
print(" ".join(cmd))
subprocess.run(cmd, check=True, cwd=str(APP))


In [ ]:
import json
from pathlib import Path
latest = Path("/exp/sbnd/data/users/munjung/anomaly-detection/training/benchmarks/gputnam_iterE_inference_bench_latest.json")
report = json.loads(latest.read_text())
print("host:", report["host"], "cuda_available:", report["cuda_available"])
print(f"{'device':8} {'T':>5} {'mean_s':>10} {'s/patch':>10} {'patches/h':>10}")
for r in report["runs"]:
    print(f"{r['device']:8} {r['T']:5d} {r['mean_s']:10.3f} {r['mean_s_per_patch']:10.3f} {r['patches_per_hour']:10.1f}")
by = {(r["device"], r["T"]): r for r in report["runs"]}
for T in sorted({r["T"] for r in report["runs"]}):
    if ("cpu", T) in by and ("cuda", T) in by:
        sp = by[("cpu", T)]["mean_s"] / by[("cuda", T)]["mean_s"]
        print(f"speedup T={T}: {sp:.1f}x (CPU/GPU)")
